<a href="https://colab.research.google.com/github/oluwaseyiadegbite/-dayahead-todo/blob/main/Wildfire_%E2%80%93_Shared_Data_Preparation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

file_path = Path("/content/Algerian_forest_fires_dataset_UPDATE.csv")

with file_path.open() as file:
    for _ in range(6):
        print(file.readline().strip())

Bejaia Region Dataset
day,month,year,Temperature, RH, Ws,Rain ,FFMC,DMC,DC,ISI,BUI,FWI,Classes
01,06,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
02,06,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire
03,06,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
04,06,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire


In [2]:
import csv
from collections import Counter

with file_path.open(newline="") as file:
    rows = list(csv.reader(file))

records = [
    (line_number, [value.strip() for value in row])
    for line_number, row in enumerate(rows, start=1)
    if row and row[0].strip().isdigit()
]

print("Daily records:", len(records))
print("Labels:", Counter(row[-1] for _, row in records))

for line_number, row in records:
    if len(row) != 14:
        print("Record needing inspection, file line:", line_number)
        print(row)

Daily records: 244
Labels: Counter({'fire': 138, 'not fire': 106})
Record needing inspection, file line: 171
['14', '07', '2012', '37', '37', '18', '0.2', '88.9', '12.9', '14.6 9', '12.5', '10.4', 'fire']


In [3]:
import pandas as pd

selected_records = []

for line_number, row in records:
    selected_records.append({
        "source_line": line_number,
        "temperature": float(row[3]),
        "humidity": float(row[4]),
        "wind_speed": float(row[5]),
        "rainfall": float(row[6]),
        "label": row[-1]
    })

weather_data = pd.DataFrame(selected_records)

assert len(weather_data) == 244
assert not weather_data.isna().any().any()
assert set(weather_data["label"]) == {"fire", "not fire"}

print("Records:", len(weather_data))
print("Missing values:", weather_data.isna().sum().sum())
display(weather_data.head())

Records: 244
Missing values: 0


,source_line,temperature,humidity,wind_speed,rainfall,label
0,3,29.0,57.0,18.0,0.0,not fire
1,4,29.0,61.0,13.0,1.3,not fire
2,5,26.0,82.0,22.0,13.1,not fire
3,6,25.0,89.0,13.0,2.5,not fire
4,7,27.0,77.0,16.0,0.0,not fire


In [4]:
region = None
tracking = []

for row in rows:
    if not row:
        continue

    first_value = row[0].strip()

    if "Region Dataset" in first_value:
        region = first_value.replace(" Dataset", "").strip()

    elif first_value.isdigit():
        tracking.append({
            "region": region,
            "date": pd.Timestamp(
                year=int(row[2]),
                month=int(row[1]),
                day=int(row[0])
            )
        })

assert len(tracking) == len(weather_data)

weather_data["region"] = [item["region"] for item in tracking]
weather_data["date"] = [item["date"] for item in tracking]

print("Records by region:")
print(weather_data.groupby("region").size())
print("\nDate range:")
print(weather_data["date"].min(), "to", weather_data["date"].max())


Records by region:
region
Bejaia Region            122
Sidi-Bel Abbes Region    122
dtype: int64

Date range:
2012-06-01 00:00:00 to 2012-09-30 00:00:00


In [5]:
all_dates = sorted(weather_data["date"].unique())
test_start = all_dates[-25]

development_data = weather_data[
    weather_data["date"] < test_start
].copy()

test_data = weather_data[
    weather_data["date"] >= test_start
].copy()

assert set(development_data["source_line"]).isdisjoint(
    set(test_data["source_line"])
)
assert development_data["date"].max() < test_data["date"].min()

print("Development records:", len(development_data))
print("Final test records:", len(test_data))
print("Test period starts:", pd.Timestamp(test_start).date())


Development records: 194
Final test records: 50
Test period starts: 2012-09-06


In [7]:

weather_data.to_csv("wildfire_cleaned.csv", index=False)
development_data.to_csv("wildfire_development.csv", index=False)
test_data.to_csv("wildfire_final_test.csv", index=False)


import zipfile
from google.colab import files

with zipfile.ZipFile("wildfire_shared_data.zip", "w") as folder:
    folder.write("wildfire_cleaned.csv")
    folder.write("wildfire_development.csv")
    folder.write("wildfire_final_test.csv")

files.download("wildfire_shared_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# im using  the last 20 development days for validation
dates = sorted(development_data["date"].unique())
validation_start = dates[-20]

train_data = development_data[
    development_data["date"] < validation_start
].copy()

validation_data = development_data[
    development_data["date"] >= validation_start
].copy()

print("Training records:", len(train_data))
print("Validation records:", len(validation_data))
print("Validation starts:", pd.Timestamp(validation_start).date())

Training records: 154
Validation records: 40
Validation starts: 2012-08-17


In [9]:
# Save the shared training and validation sets
train_data.to_csv("wildfire_train.csv", index=False)
validation_data.to_csv("wildfire_validation.csv", index=False)


with zipfile.ZipFile("wildfire_training_validation.zip", "w") as folder:
    folder.write("wildfire_train.csv")
    folder.write("wildfire_validation.csv")

files.download("wildfire_training_validation.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# uhm The four measurements KNN will use
inputs = ["temperature", "humidity", "wind_speed", "rainfall"]

# Training pictures are replaced by weather records in this project
X_train = train_data[inputs]
y_train = train_data["label"]


X_validation = validation_data[inputs]
y_validation = validation_data["label"]

print("Training inputs:", X_train.shape)
print("Validation inputs:", X_validation.shape)

Training inputs: (154, 4)
Validation inputs: (40, 4)
